### load sequences

In [5]:
import pandas as pd
dataset = "INA"
import pickle
with open(f'data/{dataset}_events.txt','rb') as f:
      event_sequences = pickle.load(f)
with open(f'data/{dataset}_visits.txt','rb') as f:
      visit_sequences = pickle.load(f)
y_df = pd.read_csv(f'data/{dataset}_targets.csv', index_col=0)

In [9]:
from scripts.clinical_text_embedding_pipeline import run_pipeline
from sklearn.metrics import *
from sklearn.model_selection import StratifiedKFold
from tqdm.notebook import tqdm
import numpy as np

n_splits = 5

scores = {
    "mcc": [],
    "precision": [],
    "recall": [],
    "accuracy": [],
    "brier": [],
    "auc": []
}

print(f"\n🚀 Starting {n_splits}-Fold CV...\n")

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
selected_patient_ids = y_df.index.values
y = y_df.values.astype(np.float32).ravel()
cvfolding = tqdm(skf.split(selected_patient_ids, y), total=n_splits, desc="Folds")

for fold, (train_idx, valid_idx) in enumerate(cvfolding):

    print(f"\n========== FOLD {fold} ==========\n")

    train_ids = np.array([selected_patient_ids[i] for i in train_idx])
    valid_ids = np.array([selected_patient_ids[i] for i in valid_idx])

    train_seq = {pid: event_sequences[pid] for pid in train_ids}
    valid_seq = {pid: event_sequences[pid] for pid in valid_ids}
    
    y_train = y_df.loc[train_ids].values.astype(np.float32).ravel()
    y_valid = y_df.loc[valid_ids].values.astype(np.float32).ravel()

    # ----------------------------
    # TRAIN
    # ----------------------------
    clf, X_valid, y_valid = run_pipeline(
        train_seq=train_seq,
        valid_seq=valid_seq,
        y_train=y_train,
        y_valid=y_valid,
        mode="retain"
    )

    # ----------------------------
    # PREDICT
    # ----------------------------
    y_prob = clf.predict_proba(X_valid)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    # ----------------------------
    # METRICS
    # ----------------------------
    scores["mcc"].append(matthews_corrcoef(y_valid, y_pred))
    scores["precision"].append(precision_score(y_valid, y_pred, zero_division=0))
    scores["recall"].append(recall_score(y_valid, y_pred, zero_division=0))
    scores["accuracy"].append(accuracy_score(y_valid, y_pred))
    scores["brier"].append(brier_score_loss(y_valid, y_prob))

    try:
        scores["auc"].append(roc_auc_score(y_valid, y_prob))
    except:
        scores["auc"].append(np.nan)

    print(f"Fold {fold} MCC: {scores['mcc'][-1]:.4f}")
    print(f"Fold {fold} F1 components → P:{scores['precision'][-1]:.4f} R:{scores['recall'][-1]:.4f}")
    print(f"Fold {fold} ACC: {scores['accuracy'][-1]:.4f}")
    print(f"Fold {fold} Brier: {scores['brier'][-1]:.4f}")
    print(f"Fold {fold} AUC: {scores['auc'][-1]:.4f}")
    
print("\n==============================")
print("FINAL CV RESULTS")
print("==============================")

for k in scores.keys():
    vals = np.array(scores[k], dtype=float)
    print(f"{k.upper():10s} | mean = {np.nanmean(vals):.4f} | std = {np.nanstd(vals):.4f}")


🚀 Starting 5-Fold CV...



Folds:   0%|          | 0/5 [00:00<?, ?it/s]


========== FOLD 0 ==========



2026-05-25 08:50:47.808095: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-25 08:50:47.822488: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779691847.836938 1341822 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779691847.841079 1341822 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779691847.852443 1341822 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

[LightGBM] [Info] Number of positive: 96, number of negative: 1092
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003604 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 32640
[LightGBM] [Info] Number of data points in the train set: 1188, number of used features: 128
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080808 -> initscore=-2.431418
[LightGBM] [Info] Start training from score -2.431418
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

TypeError: cannot unpack non-iterable LGBMClassifier object